In [30]:
import sys, numpy, pandas, sklearn, matplotlib
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)

Python: 3.12.6 (v3.12.6:a4a2d2b0d85, Sep  6 2024, 16:08:03) [Clang 13.0.0 (clang-1300.0.29.30)]
NumPy: 2.0.2
Pandas: 2.3.3
scikit-learn: 1.6.1
Matplotlib: 3.9.4


id: S24073265

In [31]:
import pandas as pd

raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=3265)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

(1154, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           57
building_type         0
ceiling_height_m    133
price_kzt             0
dtype: int64
                   count unique         top freq         mean         std  \
listing_id          1154   1080   ALM-11147    3          NaN         NaN   
listed_date         1154    247  2026-04-18   12          NaN         NaN   
district            1154     48       Medeu  144          NaN         NaN   
rooms               1154      6           2  443          NaN         NaN   
area_

In [32]:
df[["listed_date", "area_m2", "price_kzt", "rooms"]].sample(10, random_state=10)

,listed_date,area_m2,price_kzt,rooms
862,2026-03-03,96.7,124040000,4
849,2026-05-27,84.0,75680000,3
1105,2026-04-19,50.2,26100000,1
273,2026-06-24,41.2,"19,530,000",1
1150,2026-05-20,114.9 m²,"105,060,000",4
987,2026-04-17,85.1,38420000,3
349,2026-08-13,31.1,18670000,studio
623,2026-04-16,46.0,33390000,2
889,2026-06-06,51.5,25020000,2
157,2026-07-07,77.0,43580000,3


Q1: Pandas read some columns as text because the values are written in different formats. For example, area can include `m²`, price can have commas or a currency sign, and rooms can contain `studio`. `listed_date` is also read as text, but it should be a date. `district` and `building_type` are categories, and `listing_id` is an ID, so those can stay as text.

In [33]:
exact_dupes = df.duplicated().sum()
print("exact copies of another row", exact_dupes)

unique_rows = df.drop_duplicates()
repeated_rows = unique_rows["listing_id"].duplicated().sum()
print("rows with a repeated listing_id after exact copies are removed:", repeated_rows)

repost_candidates = unique_rows[
    unique_rows.duplicated(subset="listing_id", keep=False)
].sort_values(["listing_id", "listed_date"])
print("listing_id values with more than one non-identical row:", repost_candidates["listing_id"].nunique())
repost_candidates

exact copies of another row 31
rows with a repeated listing_id after exact copies are removed: 43
listing_id values with more than one non-identical row: 43


,listing_id,listed_date,district,rooms,area_m2,floor,total_floors,year_built,building_type,ceiling_height_m,price_kzt
966,ALM-10003,2026-05-04,Auezov,2,48.6,2,5,1978.0,panel,2.55,28320000
7,ALM-10003,2026-05-29,Auezov,2,48.6,2,5,1978.0,panel,2.55,26120000
939,ALM-10009,2025-12-03,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,42700000
175,ALM-10009,2026-01-19,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,39400000
475,ALM-10016,2026-01-20,Auezov,1,39.7,13,25,2022.0,monolith,2.80,35130000
...,...,...,...,...,...,...,...,...,...,...,...
884,ALM-11024,2026-05-04,Nauryzbay,1,35.8,11,20,2024.0,monolith,2.87,18770000
1094,ALM-11124,2026-03-18,Nauryzbay,2,49.4,5,12,1964.0,panel,NaN,19580000
191,ALM-11124,2026-04-01,Nauryzbay,2,49.4,5,12,1964.0,panel,NaN,18680000
167,ALM-11147,2026-05-26,Almaly,2,56.7,4,9,2018.0,monolith,2.86,59610000


Q2: There are 31 exact copies. After removing them, 43 rows still have a `listing_id` that appears more than once. In the pairs I checked, the apartment details are the same but the date and price are different. I will keep the row with the latest `listed_date`, because it is the newest version of the listing.

The cleaned table should have one row for each `listing_id`.

Q3:

| Column | Expected type | Valid range |
|---|---|---|
| listing_id | text ID | non-empty; unique after reposts are removed |
| listed_date | datetime | valid dates in YYYY-MM-DD format |
| district | category | one of 8: Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu |
| rooms | int | 1–6 (`studio` will be 1) |
| area_m2 | float | about 15–250 m² |
| floor | int | 1 to `total_floors` |
| total_floors | int | 1–30 |
| year_built | int | 1930–2026; 0 is not a real year |
| building_type | category | panel / brick / monolith |
| ceiling_height_m | float | about 2.4–3.5 m; 0 is not valid |
| price_kzt | int | positive, roughly 10M–200M KZT |

One row in the cleaned table represents one apartment listing (`listing_id`), using its latest posting.

In [34]:
cleaning_log = []

def log_step(operation, before, after):
    cleaning_log.append({"operation": operation, "rows_before": before, "rows_after": after})

In [35]:
before = len(df)
df = df.drop_duplicates()
log_step("Deleting exact duplicates", before, len(df))
print(f"Was {before}, became {len(df)}")

Was 1154, became 1123


In [36]:
df["listed_date"] = pd.to_datetime(df["listed_date"])
df = df.sort_values("listed_date")

before = len(df)
df = df.drop_duplicates(subset="listing_id", keep="last")
log_step("Deleting reposts (kept the latest listing)", before, len(df))
print(f"Was {before}, became {len(df)}")

Was 1123, became 1080


In [37]:
district_map = {
    "alatau": "Alatau", "alatauskiy": "Alatau", "алатауский": "Alatau",
    "almaly": "Almaly", "almalinskiy": "Almaly", "алмалинский": "Almaly",
    "auezov": "Auezov", "auezovskiy": "Auezov", "ауэзовский": "Auezov",
    "bostandyk": "Bostandyk", "bostandykskiy": "Bostandyk", "бостандыкский": "Bostandyk",
    "medeu": "Medeu", "medeuskiy": "Medeu", "медеуский": "Medeu",
    "nauryzbay": "Nauryzbay", "nauryzbayskiy": "Nauryzbay", "наурызбайский": "Nauryzbay",
    "turksib": "Turksib", "turksibskiy": "Turksib", "турксибский": "Turksib",
    "zhetysu": "Zhetysu", "zhetysuskiy": "Zhetysu", "жетысуский": "Zhetysu",
}

key = df["district"].str.strip().str.casefold()
df["district"] = key.map(district_map)

print("Gap after mapping:", df["district"].isna().sum())
print("Unique districts:", df["district"].nunique())

Gap after mapping: 0
Unique districts: 8


In [38]:
key[df["district"].isna()].unique()

array([], dtype=object)

Step 3b: Categorical cleaning (district)

The district column had 8 districts, but they were written in different ways. There were different capital letters, extra spaces, Cyrillic names, and names ending with -skiy.

I removed extra spaces, changed all values to lowercase using casefold, and replaced different variants with the correct district name using a dictionary.

After cleaning, there are exactly 8 unique districts and no missing values.

Some rows had an empty district in the original data. These rows could not be fixed, so they were removed.

Rows removed: [enter number]


In [39]:
df["rooms"] = df["rooms"].replace({"studio": 1})
df["rooms"] = pd.to_numeric(df["rooms"], errors="coerce")

area_clean = (
    df["area_m2"].astype(str)
    .str.replace("m²", "", regex=False)
    .str.strip()
    .str.replace(",", ".", regex=False)
)
df["area_before_fix"] = pd.to_numeric(area_clean, errors="coerce")
df["area_m2"] = df["area_before_fix"]

df["price_kzt"] = df["price_kzt"].astype(str).str.replace(r"[^\d]", "", regex=True)
df["price_kzt"] = pd.to_numeric(df["price_kzt"], errors="coerce")

print("NaN rooms:", df["rooms"].isna().sum())
print("NaN area_m2:", df["area_m2"].isna().sum())
print("NaN price_kzt:", df["price_kzt"].isna().sum())

NaN rooms: 0
NaN area_m2: 0
NaN price_kzt: 0


Step 3c: Text-to-numeric

rooms: "studio" was replaced with 1 and then changed to numeric. 0 NaN values were created.

area_m2: I removed the "m²" unit and replaced commas with dots. 0 NaN values were created.

price_kzt: I removed all non-digit characters, including commas, spaces, and the ₸ symbol. 0 NaN values were created.

The comma has a different meaning in each column. In area_m2 it is a decimal separator, while in price_kzt it is a thousands separator. So each column needed a different rule.


In [41]:
import numpy as np

print((df["year_built"] == 0).sum())
print((df["ceiling_height_m"] == 0).sum())
print((df["floor"] == -1).sum())

df["year_built"] = df["year_built"].replace(0, np.nan)
df["ceiling_height_m"] = df["ceiling_height_m"].replace(0, np.nan)
df["floor"] = df["floor"].replace(-1, np.nan)

22
23
12


Step 4.1

year_built = 0: 22 rows replaced with NaN
ceiling_height_m = 0: 23 rows replaced with NaN
floor = -1: 12 rows replaced with NaN

In [42]:
impossible_floor = df["floor"] > df["total_floors"]
print(impossible_floor.sum())
df.loc[impossible_floor, ["listing_id", "floor", "total_floors"]]

12


,listing_id,floor,total_floors
855,ALM-10729,15.0,12
739,ALM-10851,13.0,12
1058,ALM-10944,4.0,3
651,ALM-10044,30.0,25
229,ALM-10420,11.0,9
1118,ALM-10636,6.0,5
667,ALM-10628,6.0,5
793,ALM-10321,10.0,5
265,ALM-10266,13.0,12
121,ALM-10881,8.0,5


In [43]:
df.loc[impossible_floor, "floor"] = np.nan
print(df["floor"].isna().sum())

24


Step 4.2

Rule: floor > total_floors is impossible.
12 rows violated this rule. Set floor to NaN in these rows (total_floors is a fixed 
building property, less likely to be a typo than a single listing's floor value).

In [44]:
df["price_per_m2"] = df["price_kzt"] / df["area_m2"]

def iqr_bounds(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

low, high = iqr_bounds(df["price_per_m2"])
global_outliers = df[(df["price_per_m2"] < low) | (df["price_per_m2"] > high)]
print("Global outliers:", len(global_outliers))

group_bounds = df.groupby("district")["price_per_m2"].apply(iqr_bounds)
def is_outlier_grouped(row):
    lo, hi = group_bounds[row["district"]]
    return row["price_per_m2"] < lo or row["price_per_m2"] > hi
grouped_outliers = df[df.apply(is_outlier_grouped, axis=1)]
print("Grouped (by district) outliers:", len(grouped_outliers))

Global outliers: 0
Grouped (by district) outliers: 12


In [45]:
grouped_outliers[["listing_id", "district", "rooms", "area_m2", "price_kzt", "price_per_m2"]]

,listing_id,district,rooms,area_m2,price_kzt,price_per_m2
898,ALM-10910,Zhetysu,3,736.0,54050000,73437.500000
989,ALM-10276,Turksib,2,604.0,33630000,55678.807947
559,ALM-10405,Turksib,1,356.0,18830000,52893.258427
1073,ALM-11122,Zhetysu,2,647.0,40670000,62859.350850
297,ALM-10072,Medeu,2,530.0,64270000,121264.150943
964,ALM-10058,Bostandyk,2,555.0,66270000,119405.405405
748,ALM-10921,Zhetysu,3,668.0,38560000,57724.550898
44,ALM-10264,Bostandyk,1,495.0,42680000,86222.222222
904,ALM-10909,Alatau,2,565.0,32610000,57716.814159
691,ALM-10440,Auezov,3,879.0,48370000,55028.441411


In [46]:
fix_mask = df["area_m2"] > 200
df.loc[fix_mask, "area_m2"] = df.loc[fix_mask, "area_m2"] / 10
df["price_per_m2"] = df["price_kzt"] / df["area_m2"]

df.loc[fix_mask, ["listing_id", "district", "area_m2", "price_kzt", "price_per_m2"]]

,listing_id,district,area_m2,price_kzt,price_per_m2
898,ALM-10910,Zhetysu,73.6,54050000,7.343750e+05
989,ALM-10276,Turksib,60.4,33630000,5.567881e+05
559,ALM-10405,Turksib,35.6,18830000,5.289326e+05
1073,ALM-11122,Zhetysu,64.7,40670000,6.285935e+05
297,ALM-10072,Medeu,53.0,64270000,1.212642e+06
964,ALM-10058,Bostandyk,55.5,66270000,1.194054e+06
748,ALM-10921,Zhetysu,66.8,38560000,5.772455e+05
44,ALM-10264,Bostandyk,49.5,42680000,8.622222e+05
904,ALM-10909,Alatau,56.5,32610000,5.771681e+05
691,ALM-10440,Auezov,87.9,48370000,5.502844e+05


Step 4.3-4.4
Global IQR on price_per_m2 found 0 outliers, but grouping by district found 12 — 
global bounds were too wide across districts with different price levels.

All 12 flagged rows had area_m2 in the 350-900 range (physically impossible for an 
apartment) and abnormally low price_per_m2. Dividing area_m2 by 10 (lost decimal point) 
brought price_per_m2 back into the normal 460k-1.2M KZT/m² range for these districts.

Decision: fixed (divided by 10), not deleted, since after correction the values are 
consistent with neighboring listings.

In [47]:
df.to_csv("data/almaty_apartments_clean.csv", index=False)

In [48]:
from sklearn.model_selection import train_test_split

NUMERIC = ["rooms", "area_m2", "floor", "total_floors", "year_built", "ceiling_height_m"]

train, test = train_test_split(df, test_size=0.2, random_state=3265)
train = train.copy()
test = test.copy()

print(train.shape, test.shape)

(864, 13) (216, 13)


In [49]:
print(train.isna().sum().sort_values(ascending=False))

ceiling_height_m    118
year_built           63
floor                20
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
total_floors          0
building_type         0
price_kzt             0
area_before_fix       0
price_per_m2          0
dtype: int64


In [50]:
print(train.groupby("building_type")["ceiling_height_m"].apply(lambda s: s.isna().mean()))
print(train.groupby("building_type")["year_built"].apply(lambda s: s.isna().mean()))

building_type
brick       0.136150
monolith    0.040678
panel       0.216292
Name: ceiling_height_m, dtype: float64
building_type
brick       0.089202
monolith    0.050847
panel       0.081461
Name: year_built, dtype: float64


Step 5b
ceiling_height_m missing rate: panel 21.6%, brick 13.6%, monolith 4.1% — rates differ 
strongly by building_type -> looks like MAR.

year_built missing rate: brick 8.9%, panel 8.1%, monolith 5.1% — rates are close -> looks 
like MCAR.

In [51]:
print(train.dropna().shape)

(677, 13)


In [52]:
from sklearn.metrics import mean_absolute_error

known = train[train["ceiling_height_m"].notna()]
hidden_idx = known.sample(frac=0.15, random_state=3265).index
rest = known.drop(hidden_idx)
answers = known.loc[hidden_idx, "ceiling_height_m"]

global_median = rest["ceiling_height_m"].median()
pred_global = pd.Series(global_median, index=hidden_idx)
mae_global = mean_absolute_error(answers, pred_global)

group_medians = rest.groupby("building_type")["ceiling_height_m"].median()
pred_group = known.loc[hidden_idx, "building_type"].map(group_medians)
mae_group = mean_absolute_error(answers, pred_group)

print("MAE global median:", mae_global)
print("MAE group median:", mae_group)

MAE global median: 0.15125000000000008
MAE group median: 0.07750000000000005


Step 5c
MAE global median: 0.151
MAE group median (by building_type): 0.078

Group median wins - ceiling height clearly differs by building type, so filling with a 
single global number loses accuracy.

In [53]:
fill_cols = ["ceiling_height_m", "year_built", "floor"]

medians_by_type = train.groupby("building_type")[fill_cols].median()

for col in fill_cols:
    train[col] = train[col].fillna(train["building_type"].map(medians_by_type[col]))
    test[col] = test[col].fillna(test["building_type"].map(medians_by_type[col]))

print(train[NUMERIC].isna().sum().sum())
print(test[NUMERIC].isna().sum().sum())

0
0


Step 5 report

**Missing values table (train):**
ceiling_height_m: 118
year_built: 63
floor: 20

**Missing rate by building_type:**
ceiling_height_m — panel 21.6%, brick 13.6%, monolith 4.1% -> rates differ strongly -> MAR
year_built — brick 8.9%, panel 8.1%, monolith 5.1% -> rates are close -> MCAR

dropna() would remove 187 of 864 rows (21.6%) — too costly, so we impute instead.

**Masking test (ceiling_height_m):**
MAE global median: 0.151
MAE group median (by building_type): 0.078
Group median wins.

**Chosen strategy per column:**
ceiling_height_m — median by building_type (masking test showed it's more accurate).
year_built — median by building_type (consistent with ceiling_height_m, and building_type 
is available for every row).
floor — median by building_type (same reasoning, keeps method consistent across columns).

**NaN remaining after fill:**
train: 0
test: 0